In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker


In [ ]:
df = pd.read_pickle("instagram_node_dataframe.pkl")
df.head()

### Datenaufbereitung für die Analyse
Hier werden für die Analyse relevante Daten aufbereitet und passendere Spaltennamen vergeben. 

Im df_gefiltert Dataframe enthält die wichtigsten Spalten, die einen Einfluss auf Likeanzahl haben könnten. 


In [ ]:
df_gefiltert = df.rename(columns={
    "edge_media_preview_like.count": "likes",
    "owner.edge_followed_by.count": "follower_count",
    "owner.is_verified": "is_verified",
    "owner.category_name": "account_category",
    "owner.is_business_account": "is_business_account",
    "owner.is_professional_account": "is_professional_account",
    "edge_media_to_comment.count": "comment_count",
    "edge_media_to_tagged_user.edges": "tagged_users",
    "edge_media_to_caption.edges": "caption_text",
    "__typename": "post_type",
    "edge_sidecar_to_children.edges": "is_carousel",
    "taken_at_timestamp": "timestamp"
})

df_gefiltert = df_gefiltert[[
    "likes",
    "follower_count",
    "is_verified",
    "account_category",
    "is_business_account",
    "is_professional_account",
    "comment_count",
    "tagged_users",
    "caption_text",
    "post_type",
    "is_carousel",
    "timestamp"
]]
df_gefiltert

In [ ]:
df_gefiltert.info()

## Hypothesen

#### 1. Hypothese: Accounts mit mehr Followern (`follower_count`) erzielen tendenziell mehr Likes (`likes`).
      - H0: Es gibt keinen (linearen/monotonen) Zusammenhang zwischen Follower-Anzahl und Likes.
      - H1: Es gibt einen (linearen/monotonen) Zusammenhang zwischen Follower-Anzahl und Likes.

In [ ]:
df_test1 = plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='follower_count',
    y='likes',
    data=df_gefiltert,
    alpha=0.5
)
plt.title('Streudiagramm: Follower-Anzahl vs. Likes')
plt.xlabel('Anzahl Follower des Posters')
plt.ylabel('Anzahl Likes')
plt.xscale('log')
plt.yscale('log')
plt.grid(True, which="both", ls="--", linewidth=0.3)
plt.gca().xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{int(y):,}'))
plt.show()

##### Visuelle Inspektion: Streudiagramm `follower_count` vs. `likes`

Das erstellte Streudiagramm visualisiert den Zusammenhang zwischen der Anzahl der Follower eines Posters (X-Achse) und der Anzahl der Likes, die ein Post erhalten hat (Y-Achse).

**Beobachtungen aus dem Diagramm:**

* Es ist eine **positive Tendenz** erkennbar: Mit steigender Follower-Anzahl (Bewegung nach rechts auf der X-Achse) ist auch eine Tendenz zu höheren Like-Zahlen (Bewegung nach oben auf der Y-Achse) zu beobachten.
* Die Mehrheit der Posts konzentriert sich im unteren linken Bereich, was auf viele Posts von Accounts mit weniger Followern und entsprechend weniger Likes hindeutet.
* Es gibt eine **starke Streuung** der Like-Zahlen, besonders bei Accounts mit einer mittleren bis hohen Follower-Anzahl. Das bedeutet, dass nicht alle Accounts mit vielen Followern automatisch extrem viele Likes für jeden Post erhalten.
* Einige wenige Accounts mit sehr hohen Follower-Zahlen (im rechten Bereich der X-Achse) erzielen auch die höchsten Like-Zahlen im Datensatz.



In [ ]:
# Berechnung der Spearman-Rangkorrelation
correlation_coefficient, p_value = stats.spearmanr(df_gefiltert['follower_count'], df_gefiltert['likes'])

print(f"\n--- Spearman-Rangkorrelation ---")
print(f"Korrelationskoeffizient: {correlation_coefficient:.4f}")
print(f"P-Wert: {p_value:.4g}") 

# Testentscheidung
alpha = 0.05
print(f"Signifikanzniveau (alpha): {alpha}")
if p_value < alpha:
    print(f"Entscheidung: Die Nullhypothese (H0) wird abgelehnt (p-Wert = {p_value:.4g} < {alpha}).")
    if correlation_coefficient > 0:
        print("Interpretation: Es besteht ein statistisch signifikanter positiver monotoner Zusammenhang.")
    elif correlation_coefficient < 0:
        print("Interpretation: Es besteht ein statistisch signifikanter negativer monotoner Zusammenhang.")
    else:
        print("Interpretation: Es besteht ein statistisch signifikanter Zusammenhang, aber der Koeffizient ist nahe Null.")
else:
    print(f"Entscheidung: Die Nullhypothese (H0) kann nicht abgelehnt werden (p-Wert = {p_value:.4g} >= {alpha}).")
    print("Interpretation: Es konnte kein statistisch signifikanter monotoner Zusammenhang nachgewiesen werden.")


#### Interpretation der Ergebnisse für Hypothese 1: Einfluss der Follower-Anzahl auf Likes

Die statistische Überprüfung des Zusammenhangs zwischen der Anzahl der Follower eines Posters (`follower_count`) und der Anzahl der Likes (`likes`) für deren Posts ergab folgende Resultate:

* **Spearman-Rangkorrelationskoeffizient ($\rho$):** 0.7642
* **P-Wert:** 0.0 (genauer gesagt ein sehr kleiner Wert, der als 0 ausgegeben wird)
* **Signifikanzniveau ($\alpha$):** 0.05

**Testentscheidung:**

Da der berechnete **p-Wert (0.0) kleiner ist als das festgelegte Signifikanzniveau ($\alpha = 0.05$)**, wird die Nullhypothese ($H_0$) abgelehnt. Die Nullhypothese besagte, dass kein monotoner Zusammenhang zwischen der Follower-Anzahl und den Likes besteht.

**Interpretation im Kontext der Hypothese:**

Das Ergebnis stützt die Alternativhypothese ($H_1$) und die ursprüngliche Forschungshypothese: Es besteht ein **statistisch signifikanter, positiver monotoner Zusammenhang** zwischen der Anzahl der Follower eines Instagram-Accounts und der Anzahl der Likes, die dessen Posts erhalten.

* Der **Korrelationskoeffizient von $\rho \approx 0.76$** deutet auf einen **starken positiven Zusammenhang** hin. Das bedeutet, dass tendenziell Accounts mit einer höheren Follower-Anzahl auch mehr Likes auf ihre Posts bekommen. Umgekehrt erhalten Accounts mit weniger Followern tendenziell weniger Likes.
* Das **Streudiagramm** visualisiert diesen Trend ebenfalls: Obwohl es eine gewisse Streuung gibt, ist eine klare Tendenz von links unten nach rechts oben erkennbar. Besonders auffällig sind einige Accounts mit extrem hohen Follower-Zahlen, die auch sehr hohe Like-Zahlen erreichen.



2. is_verified: (true / false)
    - Hypothese: Verifizierte Accounts (blauer Haken) haben möglicherweise mehr Glaubwürdigkeit oder Reichweite, was zu mehr Likes führen könnte.
        - H0: Die durchschnittliche Like-Anzahl ist für verifizierte und nicht-verifizierte Accounts gleich.
        - H1: Die durchschnittliche Like-Anzahl ist für verifizierte und nicht-verifizierte Accounts unterschiedlich.

3. owner_category: account_category (z.B. "Motivational speaker")
    - Hypothese: Bestimmte Account-Kategorien (z.B. Mode, Reisen, Essen) könnten generell populärer sein oder unterschiedliches Engagement hervorrufen. 
        - H0: Die durchschnittliche Like-Anzahl ist über alle Account-Kategorien gleich.
        - H1: Mindestens eine Account-Kategorie hat eine andere durchschnittliche Like-Anzahl.

4. Account-Typ: Abgeleitet aus is_business_account, is_professional_account
    - Hypothese: Der Typ des Accounts könnte das Engagement beeinflussen.

5. Merkmale des Posts selbst:

    - timestamp: node.taken_at_timestamp (Unix-Timestamp, z.B. 1680008999)

    - Hypothese: Der Zeitpunkt des Posts hat oft einen großen Einfluss. Daraus zu extrahieren sind:
        - Stunde des Tages: (z.B. 16 Uhr) - Posts zu bestimmten Zeiten (abends, mittags) könnten besser performen.
        - Wochentag: (z.B. Montag=0, Sonntag=6) - Wochenenden vs. Wochentage.
        - Monat / Jahreszeit: Saisonalität.

        - H0: Die durchschnittliche Like-Anzahl ist über alle Tageszeit-Gruppen gleich.
        - H1: Mindestens eine Tageszeit-Gruppe hat eine andere durchschnittliche Like-Anzahl.


In [ ]:
df_gefiltert['datetime'] = pd.to_datetime(df_gefiltert['timestamp'], unit='s')
df_gefiltert['hour_of_day'] = df_gefiltert['datetime'].dt.hour
df_gefiltert['day_of_week'] = df_gefiltert['datetime'].dt.day_name(locale='de_DE')  # z.B. 'Montag', 'Dienstag'
df_gefiltert['month'] = df_gefiltert['datetime'].dt.month_name(locale='de_DE')      # z.B. 'Januar', 'Februar'


6. post_type: Abgeleitet aus post_type (z.B. GraphImage, GraphVideo, GraphSidecar)
    - Hypothese: Reine Bilder vs. Videos vs. Karussell-Posts könnten unterschiedlich abschneiden.

7. media_count_in_post: Anzahl der Elemente in einem Karussell (abgeleitet aus len(is_carousel), falls post_type == GraphSidecar)
    - Hypothese: Die Anzahl der Slides in einem Karussell könnte die Verweildauer und damit das Engagement beeinflussen.

In [ ]:
def check_carousel_and_count(edges):
    if isinstance(edges, list) and len(edges) > 0:
        return True, len(edges) # Ist Karussell, Anzahl der Medien
    return False, 1 # Kein Karussell (oder einzelnes Medium)

results = df_gefiltert['is_carousel'].apply(lambda x: check_carousel_and_count(x)) 
df_gefiltert['is_carousel_bool'] = [res[0] for res in results]
df_gefiltert['media_count_in_post'] = [res[1] for res in results]

# Für Nicht-Karussell-Posts ('GraphImage', (kein 'GraphSidecar/Karusell')) ist media_count_in_post typischerweise 1
df_gefiltert.loc[df_gefiltert['post_type'] != 'GraphSidecar', 'media_count_in_post'] = 1
df_gefiltert.loc[df_gefiltert['post_type'] != 'GraphSidecar', 'is_carousel_bool'] = False

8. comments: comment_count 
    - Hypothese: Posts mit vielen Kommentaren haben oft auch viele Likes (Korrelation). Als direktes Feature für die Vorhersage von Likes ist es aber schwierig, da Kommentare oft nach den Likes kommen. Es ist aber ein wichtiger Teil.

9. tagged_user_count: Anzahl der markierten Nutzer (abgeleitet aus len(node.edge_media_to_tagged_user.edges))
    - Hypothese: Das Markieren anderer Nutzer könnte die Sichtbarkeit und Interaktion erhöhen.

10. location_tagged: Prüfen, ob node.location nicht null ist.
    - Hypothese: Das Hinzufügen eines Ortes könnte die Reichweite in bestimmten Bereichen erhöhen.

11. Merkmale aus dem Text (Caption):
    - Hypothese: Der Text könnte das Engagement beeinflussen. Längere Texte, viele Hashtags oder Erwähnungen könnten mehr Likes bringen.
    - caption_text: node.edge_media_to_caption.edges[0].node.text
Hieraus lassen sich weitere Merkmale extrahieren:
        - caption_length: Länge des Textes.
        - hashtag_count: Anzahl der Hashtags (#) im Text.
        - mention_count: Anzahl der Erwähnungen (@) im Text.





In [ ]:
# caption_text Extraktion
def extract_caption(edges):
    if isinstance(edges, list) and len(edges) > 0 and 'node' in edges[0] and 'text' in edges[0]['node']:
        return edges[0]['node']['text']
    return "" 

df_gefiltert['actual_caption_text'] = df_gefiltert['caption_text'].apply(extract_caption)
df_gefiltert['caption_length'] = df_gefiltert['actual_caption_text'].apply(len)
df_gefiltert['hashtag_count'] = df_gefiltert['actual_caption_text'].str.count('#')
df_gefiltert['mention_count'] = df_gefiltert['actual_caption_text'].str.count('@')


In [ ]:
pd.DataFrame(df_gefiltert.describe()).round(2)